In [ ]:
from pathlib import Path
import warnings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Project paths
PROJECT_DIR = Path(__file__).resolve().parent.parent
DATA_PATH = PROJECT_DIR / "data" / "Sample - Superstore.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs"

In [ ]:
def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    # 1. Load the dataset
    df = pd.read_csv(DATA_PATH, parse_dates=["Order_Date"])

    # Provisional assumption: delivered orders approximate completed sales.
    delivered = df[df["Order_Status"] == "Delivered"].copy()

    # 2. Calculate monthly delivered net sales
    monthly_sales = (
        delivered.set_index("Order_Date")
        .resample("MS")["Net_Sales"]
        .sum()
        .sort_index()
    )

    print("DELIVERED ORDERS:", len(delivered))
    print("MONTHS IN SERIES:", len(monthly_sales))
    print(
        "MONTHS WITH NO DELIVERED SALES:",
        int((monthly_sales == 0).sum())
    )

    # 3. Split the data: use the final six months for testing
    if len(monthly_sales) < 18:
        raise ValueError(
            "At least 18 months of data are needed for this forecast setup."
        )

    train = monthly_sales.iloc[:-6]
    test = monthly_sales.iloc[-6:]

    # 4. Train the seasonal forecasting model
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        model = ExponentialSmoothing(
            train,
            trend="add",
            seasonal="add",
            seasonal_periods=12,
            initialization_method="estimated",
        ).fit()

    # 5. Evaluate the model against the actual test data
    predictions = model.forecast(len(test))

    mae = mean_absolute_error(test, predictions)
    rmse = np.sqrt(mean_squared_error(test, predictions))

    nonzero_test = test != 0

    if nonzero_test.any():
        mape = (
            (
                test[nonzero_test] - predictions[nonzero_test]
            ).abs()
            / test[nonzero_test].abs()
        ).mean() * 100
    else:
        mape = float("nan")

    # 6. Seasonal-naive baseline:
    # predict each test month using the same month from the previous year.
    baseline_dates = test.index - pd.DateOffset(years=1)
    baseline_predictions = monthly_sales.loc[baseline_dates].to_numpy()

    baseline_mae = mean_absolute_error(test, baseline_predictions)
    baseline_rmse = np.sqrt(
        mean_squared_error(test, baseline_predictions)
    )

    print("\nFORECAST MODEL EVALUATION")
    print(
        f"Test period: {test.index.min().date()} "
        f"to {test.index.max().date()}"
    )
    print(f"Model MAE:  {mae:.2f}")
    print(f"Model RMSE: {rmse:.2f}")
    print(f"Model MAPE: {mape:.2f}%")
    print(f"Seasonal baseline MAE:  {baseline_mae:.2f}")
    print(f"Seasonal baseline RMSE: {baseline_rmse:.2f}")

    # Save actual values and predictions for comparison
    comparison = pd.DataFrame(
        {
            "Actual_Net_Sales": test.to_numpy(),
            "Model_Prediction": predictions.to_numpy(),
            "Seasonal_Baseline": baseline_predictions,
        },
        index=test.index,
    )
    comparison.index.name = "Month"
    comparison.to_csv(OUTPUT_DIR / "forecast_model_evaluation.csv")

    # 7. Train the final model on all available monthly data
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        final_model = ExponentialSmoothing(
            monthly_sales,
            trend="add",
            seasonal="add",
            seasonal_periods=12,
            initialization_method="estimated",
        ).fit()

    # 8. Forecast the next six months
    future_forecast = final_model.forecast(6)
    future_forecast.name = "Forecast_Net_Sales"

    future_forecast.to_csv(
        OUTPUT_DIR / "sales_forecast_next_6_months.csv",
        index_label="Month",
    )

    # 9. Plot historical sales, test predictions, and future forecast
    plt.figure(figsize=(12, 6))

    plt.plot(
        monthly_sales.index,
        monthly_sales,
        label="Historical delivered net sales",
    )

    plt.plot(
        test.index,
        predictions,
        label="Test-period predictions",
        linestyle="--",
    )

    plt.plot(
        future_forecast.index,
        future_forecast,
        label="Next 6 months forecast",
        linestyle="--",
    )

    plt.title("Monthly Delivered Net Sales Forecast")
    plt.xlabel("Month")
    plt.ylabel("Net Sales")
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / "sales_forecast.png",
        dpi=300,
    )
    plt.close()

    # 10. Display the forecast
    print("\nNEXT SIX MONTHS FORECAST")
    print(future_forecast.round(2).to_string())

    print("\nFiles saved in the outputs folder:")
    print("- forecast_model_evaluation.csv")
    print("- sales_forecast_next_6_months.csv")
    print("- sales_forecast.png")

    print(
        "\nCaution: This dataset is synthetic. Delivered Net_Sales is "
        "being used as a proxy for completed sales; the accounting "
        "treatment of returns should be verified before business use."
    )

In [ ]:
if __name__ == "__main__":
    main()